In [59]:
import torch
print(torch.__version__)

2.14.0+cpu


In [60]:
from pathlib import Path
from PIL import Image
from numpy import asarray

# x_data = []
# y_target = []
# mnist_path = Path('MNIST')
# n = 0
# for digit_folder in sorted(mnist_path.glob('*')):
#     if digit_folder.is_dir():
#         # Get all PNGs in this folder, sort them, and take first 1000
#         images = sorted(digit_folder.glob('*.png'))[:1000]
#         for image_path in images:
#             img = Image.open(image_path)
#             pixel_array = asarray(img)
#             flat_image = pixel_array.ravel()
#             x_data.append(flat_image)
#             y_target.append(n)
#     n += 1

In [61]:
import torch.nn as nn
import torchvision as tv
from torch.utils.data import TensorDataset, DataLoader

# imports
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from PIL import Image
from numpy import asarray
import pandas as pd
from pathlib import Path
import numpy as np

In [62]:
# x_data_array = np.array(x_data)
# np.savez_compressed('x_short_data.npz', data=x_data_array)
# y_target_array = np.array(y_target)
# np.savez_compressed('y_short_target.npz', data=y_target_array)

In [63]:
# To reload x and y:
loaded = np.load('x_short_data.npz')
x_data = loaded['data']

loaded = np.load('y_short_target.npz')
y_target = loaded['data']

In [64]:
print(x_data[0])

[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0  51 159 253 159  50   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0  48 238 252 252 252 237   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0  54 227 253 252 239 233 252  57   6   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0  10  60 224 252 253 252 202  84 252
 253 122   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0 163 252 252 252 253 252 252  96 189 253 167   

In [65]:
# normalize 
x_data = x_data / 255

In [66]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_target, test_size=0.3,shuffle=True)

In [67]:
x_train_t = torch.as_tensor(x_train, dtype=torch.float32).reshape(-1, 1, 28, 28)
x_test_t = torch.as_tensor(x_test, dtype=torch.float32).reshape(-1, 1, 28, 28)
y_train_t = torch.as_tensor(y_train, dtype=torch.long)
y_test_t = torch.as_tensor(y_test, dtype=torch.long)

train_dl = DataLoader(TensorDataset(x_train_t, y_train_t), batch_size=64, shuffle=True)
valid_dl = DataLoader(TensorDataset(x_test_t, y_test_t), batch_size=64)

In [68]:
model = nn.Sequential()
model.add_module(
    'conv1',
    nn.Conv2d(
        in_channels=1,
        padding=2,
        kernel_size=5,
        out_channels=32
    )
)

In [69]:

model.add_module('relu1', nn.ReLU())
model.add_module('pool1', nn.MaxPool2d(kernel_size=2))
model.add_module(
    'conv2',
    nn.Conv2d(
        in_channels=32,
        out_channels=64,
        kernel_size=5,
        padding=2
    )
)
model.add_module('relu2', nn.ReLU())
model.add_module('pool2', nn.MaxPool2d(kernel_size=2))

In [70]:
# x = torch.ones((4, 1, 28, 28))
# model(x).shape

In [71]:
model.add_module('flatten', nn.Flatten())
x = torch.ones((4,1,28,28))
model(x).shape

torch.Size([4, 3136])

In [72]:
model.add_module('fc1', nn.Linear(3136,1024))
model.add_module('relu3', nn.ReLU())
model.add_module('dropout', nn.Dropout(p=0.5))
model.add_module('fc2', nn.Linear(1024, 10))

In [73]:
loss_fn = nn.CrossEntropyLoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=0.001)

In [74]:
def train(model, num_epochs, train_dl, valid_dl):
    loss_hist_train = [0] * num_epochs
    accuracy_hist_train = [0] * num_epochs
    loss_hist_valid = [0] * num_epochs
    accuracy_hist_valid = [0] * num_epochs

    for epoch in range(num_epochs):
        model.train()
        for x_batch, y_batch in train_dl:
            pred = model(x_batch)
            loss = loss_fn(pred, y_batch)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            loss_hist_train[epoch] += loss.item() * y_batch.size(0)
            is_correct = (
                torch.argmax(pred, dim=1) == y_batch
            ).float()
            accuracy_hist_train[epoch] += is_correct.sum()
        loss_hist_train[epoch] /= len(train_dl.dataset)
        accuracy_hist_train[epoch] /= len(train_dl.dataset)
        model.eval()
        
        with torch.no_grad():
            for x_batch, y_batch in valid_dl:
                pred = model(x_batch)
                loss = loss_fn(pred, y_batch)
                loss_hist_valid[epoch] += loss.item() * y_batch.size(0)
                is_correct = (
                    torch.argmax(pred,dim=1) == y_batch
                ).float()
                accuracy_hist_valid[epoch] += is_correct.sum()
                
        loss_hist_valid[epoch] /= len(valid_dl.dataset)
        accuracy_hist_valid[epoch] /= len(valid_dl.dataset)
        
        print(f'Epoch {epoch+1} accuracy: ')
        print(f'{accuracy_hist_train[epoch]:.4f} val_accuracy: ')
        print(f'{accuracy_hist_valid[epoch]:.4f}')
        
    return loss_hist_train, loss_hist_valid, accuracy_hist_train, accuracy_hist_valid

In [75]:
torch.manual_seed(1)
num_epochs = 20
hist = train(model, num_epochs, train_dl, valid_dl)

Epoch 1 accuracy: 
0.8274 val_accuracy: 
0.9380
Epoch 2 accuracy: 
0.9539 val_accuracy: 
0.9657
Epoch 3 accuracy: 
0.9733 val_accuracy: 
0.9710
Epoch 4 accuracy: 
0.9789 val_accuracy: 
0.9767
Epoch 5 accuracy: 
0.9844 val_accuracy: 
0.9777
Epoch 6 accuracy: 
0.9867 val_accuracy: 
0.9770
Epoch 7 accuracy: 
0.9891 val_accuracy: 
0.9780
Epoch 8 accuracy: 
0.9914 val_accuracy: 
0.9780
Epoch 9 accuracy: 
0.9921 val_accuracy: 
0.9790
Epoch 10 accuracy: 
0.9929 val_accuracy: 
0.9730
Epoch 11 accuracy: 
0.9949 val_accuracy: 
0.9763
Epoch 12 accuracy: 
0.9967 val_accuracy: 
0.9783
Epoch 13 accuracy: 
0.9983 val_accuracy: 
0.9763
Epoch 14 accuracy: 
0.9967 val_accuracy: 
0.9800
Epoch 15 accuracy: 
0.9947 val_accuracy: 
0.9823
Epoch 16 accuracy: 
0.9959 val_accuracy: 
0.9780
Epoch 17 accuracy: 
0.9953 val_accuracy: 
0.9787
Epoch 18 accuracy: 
0.9964 val_accuracy: 
0.9810
Epoch 19 accuracy: 
0.9973 val_accuracy: 
0.9793
Epoch 20 accuracy: 
0.9983 val_accuracy: 
0.9823


In [76]:
# def conv1D(x, w, p=0, s=1):
#     # x = data, w = weight, p = padding, s = stride
#     w_rot = np.array(w[::-1]) # reverses the w array
#     x_padded = np.array(x)
#     if p > 0: #
#         zero_pad = np.zeros(shape=p)
#         x_padded = np.concatenate([
#             zero_pad, x_padded, zero_pad
#         ])
#     res = []
#     for i in range(0, int ((len(x_padded) - len(w_rot))) + 1, s):
#         res.append(np.sum(x_padded[i:i+w_rot.shape[0]] * w_rot))
#     return np.array(res)

# x_data = [1, 3, 2, 4, 5, 6, 1, 3]
# weight = [1, 0, 3, 1, 2]
# print('Conv1D Implementation:', conv1D(x_data, weight, p=2, s=1))
# print('NumPy Results:', np.convolve(x_data, weight, mode='same'))

In [77]:
# print(weight[::-1])

In [78]:
# loss_func = nn.BCELoss()
# loss = loss_func(torch.tensor([0.9]), torch.tensor([1.0]))
# ls_lambda = 0.001
# conv_layer = nn.Conv2d(in_channels=3, out_channels=5, kernel_size=5)
# l2_penalty = l2_lambda * sum(
#     [(p**2).sum() for p in conv_layer.parameters()]
# )
# loss_with_penalty = loss + l2_penalty
# linear_layer = nn.Linear(10,16)

# l2_penalty = l2_lambda * sum(
#     [(p**2).sum() for p in linear_layer.parameters()]
# )
# loss_with_penalty = loss + l2_penalty
